In [17]:
#!/usr/bin/env python3
"""
Ручное выравнивание траектории vSLAM с эталоном.
Масштаб и угол поворота задаются гиперпараметрами вручную.
Начальные точки совмещаются автоматически в (0, 0).
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 🎛️ ГИПЕРПАРАМЕТРЫ ВЫРАВНИВАНИЯ (НАСТРОЙТЕ ЗДЕСЬ)
# =============================================================================
# Коэффициент масштабирования vSLAM (1.0 = без изменений)
SCALE_FACTOR = 1.0       

# Угол поворота vSLAM в градусах (положительный = против часовой стрелки)
ROTATION_ANGLE_DEG = 20.0 

# Пути к файлам
XLSX_FILE = 'output/gps_data_with_meters.xlsx'
CSV_FILE = 'output/merged_trajectory.csv'
OUTPUT_DIR = 'output'
OUTPUT_PLOT = f"manual_alignment.png"
# =============================================================================


def load_reference(xlsx_path):
    """Загружает эталонную GPS-траекторию."""
    df = pd.read_excel(xlsx_path)
    print(f"📂 Эталон: {os.path.abspath(xlsx_path)} ({len(df)} точек)")
    return df['x_m'].values, df['z_m'].values


def load_vslam(csv_path):
    """Загружает траекторию vSLAM."""
    df = pd.read_csv(csv_path)
    print(f"📂 vSLAM: {os.path.abspath(csv_path)} ({len(df)} точек)")
    # Поддержка разных названий колонок
    if 'x_meters' in df.columns:
        return df['x_meters'].values, df['z_meters'].values
    elif 'x' in df.columns:
        return df['x'].values, df['z'].values
    else:
        raise ValueError(f"Не найдены колонки координат. Доступные: {list(df.columns)}")


def apply_manual_transform(x, z, scale, angle_deg):
    """
    Применяет ручной масштаб и поворот, затем смещает начало в (0,0).
    """
    # 1. Смещаем исходную траекторию так, чтобы она начиналась в (0,0)
    x_centered = x - x[0]
    z_centered = z - z[0]
    
    # 2. Применяем масштаб
    x_scaled = x_centered * scale
    z_scaled = z_centered * scale
    
    # 3. Применяем поворот вокруг начала координат
    angle_rad = np.deg2rad(angle_deg)
    cos_a = np.cos(angle_rad)
    sin_a = np.sin(angle_rad)
    
    x_rot = x_scaled * cos_a - z_scaled * sin_a
    z_rot = x_scaled * sin_a + z_scaled * cos_a
    
    return x_rot, z_rot


def plot_trajectories(x_ref, z_ref, x_vslam_aligned, output_dir):
    """Строит график сравнения."""
    os.makedirs(output_dir, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Эталон
    ax.plot(x_ref, z_ref, 'g--', linewidth=2.5, label='Эталон (GPS)', zorder=3)
    # Выровненная vSLAM
    ax.plot(x_vslam_aligned[0], x_vslam_aligned[1], 'b-', linewidth=2.5,
            label=f'vSLAM (scale={SCALE_FACTOR}, rot={ROTATION_ANGLE_DEG}°)', zorder=2)
    
    # Старт и финиш эталона
    ax.scatter([x_ref[0]], [z_ref[0]], color='lime', s=250, marker='^',
               edgecolors='black', linewidths=2, label='Старт', zorder=5)
    ax.scatter([x_ref[-1]], [z_ref[-1]], color='red', s=250, marker='v',
               edgecolors='black', linewidths=2, label='Финиш', zorder=5)
    
    ax.scatter([x_ref[8]], [z_ref[8]], color='red', s=125, marker='o',
                        edgecolors='black', linewidths=2, label='Светофор', zorder=5)
    
    ax.set_title('Ручное выравнивание траекторий', fontsize=16, fontweight='bold')
    ax.set_xlabel('X (метры)')
    ax.set_ylabel('Z (метры)')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.axis('equal')
    ax.legend(fontsize=12, loc='best')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, OUTPUT_PLOT), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ График сохранён: {os.path.abspath(os.path.join(output_dir, OUTPUT_PLOT))}")


def main():
    print(f"\n️  Ручное выравнивание:")
    print(f"   Масштаб: {SCALE_FACTOR}")
    print(f"   Поворот: {ROTATION_ANGLE_DEG}°\n")
    
    # Загрузка данных
    x_ref, z_ref = load_reference(XLSX_FILE)
    x_vslam, z_vslam = load_vslam(CSV_FILE)
    
    # Применение ручной трансформации
    x_aligned, z_aligned = apply_manual_transform(
        x_vslam, z_vslam, SCALE_FACTOR, ROTATION_ANGLE_DEG
    )
    
    # Построение графика
    plot_trajectories(x_ref, z_ref, (x_aligned, z_aligned), OUTPUT_DIR)
    
    # Сохранение результата
    aligned_csv = os.path.join(OUTPUT_DIR, 'vslam_manual_aligned.csv')
    pd.DataFrame({
        'x_meters_raw': x_vslam,
        'z_meters_raw': z_vslam,
        'x_meters_aligned': x_aligned,
        'z_meters_aligned': z_aligned
    }).to_csv(aligned_csv, index=False)
    print(f"✅ Траектория сохранена: {os.path.abspath(aligned_csv)}")
    print(f"\n🎉 Готово! Измените SCALE_FACTOR и ROTATION_ANGLE_DEG для новой попытки.")


if __name__ == '__main__':
    main()


️  Ручное выравнивание:
   Масштаб: 1.0
   Поворот: 20.0°

📂 Эталон: c:\Users\user\Desktop\регистратор\output\gps_data_with_meters.xlsx (61 точек)
📂 vSLAM: c:\Users\user\Desktop\регистратор\output\merged_trajectory.csv (79 точек)
✅ График сохранён: c:\Users\user\Desktop\регистратор\output\manual_alignment.png
✅ Траектория сохранена: c:\Users\user\Desktop\регистратор\output\vslam_manual_aligned.csv

🎉 Готово! Измените SCALE_FACTOR и ROTATION_ANGLE_DEG для новой попытки.
